In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,DataCollatorForSeq2Seq,TrainingArguments,Trainer

In [2]:
ds = load_dataset('llm-wizard/alpaca-gpt4-data-zh')
ds

Generating train split:   0%|          | 0/48818 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 48818
    })
})

In [7]:
ds = ds['train'].train_test_split(200)
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 48618
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 200
    })
})

In [4]:
ds['train'][:3]

{'instruction': ['保持健康的三个提示。', '三原色是什么？', '描述原子的结构。'],
 'input': ['', '', ''],
 'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '三原色通常指的是红色、绿色和蓝色（RGB）。它们是通过加色混合原理创建色彩的三种基础颜色。在以发光为基础的显示设备中（如电视、计算机显示器、智能手机和平板电脑显示屏）, 三原色可混合产生大量色彩。其中红色和绿色可以混合生成黄色，红色和蓝色可以混合生成品红色，蓝色和绿色可以混合生成青色。当红色、绿色和蓝色按相等比例混合时，可以产生白色或灰色。\n\n此外，在印刷和绘画中，三原色指的是以颜料为基础的红、黄和蓝颜色（RYB）。这三种颜色用以通过减色混合原理来创建色彩。不过，三原色的具体定义并不唯一，不同的颜色系统可能会采用不同的三原色。',
  '原子是物质的基本单位，它由三种基本粒子组成：质子、中子和电子。质子和中子形成原子核，位于原子中心，核外的电子围绕着原子核运动。\n\n原子结构具有层次性。原子核中，质子带正电，中子不带电（中性）。原子核非常小且致密，占据了原子总质量的绝大部分。电子带负电，通常围绕核运动，形成若干层次，称为壳层或电子层。电子数量与质子数量相等，使原子呈电中性。\n\n电子在每个壳层中都呈规律分布，并且不同壳层所能容纳的电子数也不同。在最里面的壳层一般只能容纳2个电子，其次一层最多可容纳8个电子，再往外的壳层可容纳的电子数逐层递增。\n\n原子核主要受到两种相互作用力的影响：强力和电磁力。强力的作用范围非常小，主要限制在原子核内，具有极强的吸引作用，使核子（质子和中子）紧密结合在一起。电磁力的作用范围较大，主要通过核外的电子与原子核相互作用，发挥作用。\n\n这就是原子的基本结构。原子内部结构复杂多样，不同元素的原子核中质子、中子数量不同

In [5]:
tokenizer = AutoTokenizer.from_pretrained('Langboat/bloom-389m-zh')
tokenizer

BloomTokenizerFast(name_or_path='Langboat/bloom-389m-zh', vocab_size=42437, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [8]:
tokenizer('你好')

{'input_ids': [11877], 'attention_mask': [1]}

In [9]:
def process_func(example):
    
    MAX_LENGTH = 256
    input_ids , attention_mask ,labels = [],[],[]
    instraction = tokenizer('\n'.join(['Human: '+example['instruction'],example['input']]).strip()+'\n\nAssistant: ')
    response = tokenizer(example['output']+ tokenizer.eos_token)
    
    input_ids = instraction['input_ids']+response['input_ids']
    attention_mask = instraction['attention_mask']+response['attention_mask']
    labels = [-100]*len(instraction['input_ids']) + response['input_ids']
    
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [11]:
tokenized_ds = ds.map(process_func)
tokenized_ds

Map:   0%|          | 0/48618 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 48618
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})

In [12]:
tokenizer.decode(tokenized_ds['train'][0]['input_ids'])

'Human: 火车即将出发，你发现自己没有车票。\n\nAssistant: 作为一个人工智能助手，我并不拥有物体形态，也无需乘坐交通工具。所以我不会有没有车票，也无法持有车票。如果你发现自己没有车票，您可以在车站的售票窗口或自动售票机上购买车票，或者使用手机购票软件购买电子车票。请注意，搭乘火车前务必持有有效车票，以免受到罚款或其他不必要的麻烦。</s>'

In [13]:
tokenizer.decode(list(filter(lambda x: x!=-100,tokenized_ds['train'][0]['labels'])))

'作为一个人工智能助手，我并不拥有物体形态，也无需乘坐交通工具。所以我不会有没有车票，也无法持有车票。如果你发现自己没有车票，您可以在车站的售票窗口或自动售票机上购买车票，或者使用手机购票软件购买电子车票。请注意，搭乘火车前务必持有有效车票，以免受到罚款或其他不必要的麻烦。</s>'

In [14]:
model = AutoModelForCausalLM.from_pretrained('Langboat/bloom-389m-zh')

In [15]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=2
)

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'].select(range(100)),
    eval_dataset=tokenized_ds['test'].select(range(100)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_90490/768993002.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [19]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=6, training_loss=2.480668862660726, metrics={'train_runtime': 350.9704, 'train_samples_per_second': 0.57, 'train_steps_per_second': 0.017, 'total_flos': 64603179712512.0, 'train_loss': 2.480668862660726, 'epoch': 1.6400000000000001})

In [22]:
from transformers import pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)
ipt = "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: "
pipe(ipt, max_length=512, do_sample=True, )

Device set to use mps:0


[{'generated_text': 'Human: 考试有哪些技巧？\n\nAssistant: 1. 提前准备：准备要的东西\n 2. 选择题正确率高\n 3. 选择题正确率高，且答案正确\n 4. 选择题正确率高者解答题通常需要经过思考，且正确率高者得分通常相对较高\n 5. 选择题正确率高而答案正确者解答题通常需要经过分析，且正确率高者得分通常相对较低\n 6. 选择题正确率高者解答题通常需要经过训练，且正确率高者得分通常相对较高\n7. 选择题正确率高且答案正确者解答题通常需要经过实践，且正确率高者得分通常相对较低\n8. 选择题正确率高者解答题通常需要经过尝试，且正确率高者得分通常相对较高\n9. 选择题正确率高且答案正确者解答题通常需要经过讨论，且正确率高者得分通常相对较低\n10. 选择题正确率高者解答题通常需要经过讨论，且正确率高者得分通常相对较低\n11. 选择题正确率高者解答题通常需要经过讨论，且正确率高者得分通常相对较低\n12. 选择题正确率高者解答题通常需要经过讨论，且正确率高者得分通常相对较低\n13. 选择题正确率高者解答题通常需要经过讨论，且正确率高者得分通常相对较低\n14. 选择题正确率高者解答题通常需要经过讨论，且正确率高者得分通常相对较低\n15. 选择题正确率高且答案正确者解答题通常需要经过分析，且正确率高者得分通常相对较高\n16. 选择题正确率高且答案正确者解答题通常需要经过思考，且正确率高者得分通常相对较低\n17. 选择题正确率高者解答题通常需要经过思考，且正确率高者得分通常相对较低\n18. 选择题正确率高者解答题通常需要经过思考，且正确率高者得分通常相对较低\n19. 选择题正确率高者解答题通常需要经过思考，且正确率高者得分通常相对较低\n20. 选择题正确率高者解答题通常需要经过思考，且正确率高者得分通常相对较低'}]